In [ ]:
import numpy as np
import sysl.symbolic as sls
import geolipi.symbolic as gls
from sysl.shader.evaluate_singlepass import evaluate_singlepass
from sysl.shader_runtime.generate_shader_html import create_shader_html, make_jupyter_compatible_html
from IPython.display import display, HTML


In [ ]:
# TODO: Test all combinators. 
primitive_1 = gls.Translate3D(gls.Cuboid3D((0.2,0.4, 0.8)), (-0.25, -0.5, -0.25,))
primitive_2 = gls.Translate3D(gls.Cuboid3D((0.2,0.4, 0.8,)), (0.25, 0.5, 0.25,))
prim_3 = gls.Translate3D(gls.Cuboid3D((0.25, 0.25, 0.25,)), (0.5, 0.5, 0.5))
prim_4 = gls.Translate3D(gls.Cuboid3D((0.125, 0.135, 0.115,)), (0.5, 0.5, 0.0))

comb_exprs = [
    gls.Union(primitive_1, primitive_2),
    gls.Intersection(primitive_1, primitive_2),
    gls.Difference(primitive_1, primitive_2),
    gls.Intersection(gls.Complement(primitive_1), primitive_2),
    gls.SwitchedDifference(primitive_1, primitive_2),
    gls.SmoothUnion(primitive_1, primitive_2, 0.1),
    gls.SmoothIntersection(primitive_1, primitive_2, 0.1),
    gls.SmoothDifference(primitive_1, primitive_2, 0.1),
    gls.XOR(primitive_1, primitive_2),
    
    # gls.NarySmoothUnion(primitive_1, primitive_2, 0.1),
    # gls.NarySmoothIntersection(primitive_1, primitive_2, 0.1),
    # Translations, 
    gls.Union(primitive_1, 
        gls.Translate3D(primitive_2, (0.25, 0.0, 0.0,)),
        gls.Translate3D(primitive_2, (0.0, 0.35, 0.0,)),
        gls.Translate3D(primitive_2, (0.0, 0.0, 0.45,)),
        ),
    gls.Union(primitive_1, 
        gls.EulerRotate3D(primitive_2, (0.25, 0.0, 0.0,)),
        gls.EulerRotate3D(primitive_2, (0.0, 0.35, 0.0,)),
        gls.EulerRotate3D(primitive_2, (0.0, 0.0, 0.45,)),
        ),
    gls.Union(primitive_1, 
        gls.AxisAngleRotate3D(primitive_2, (0.25, 0.0, 0.0,)),
        gls.AxisAngleRotate3D(primitive_2, (0.0, 0.35, 0.0,)),
        gls.AxisAngleRotate3D(primitive_2, (0.0, 0.0, 0.45,)),
        ),
    gls.Union(
        gls.Scale3D(primitive_1, (1.2, 1.0, 1.0)),
        gls.Scale3D(primitive_1, (1.0, 1.4, 1.0)),
        gls.Scale3D(primitive_1, (1.0, 1.0, 1.8)),
    ),
    # Dilate Erode
    gls.Union(primitive_1, gls.Dilate3D(primitive_2, (0.1,))),
    gls.Union(primitive_1, gls.Erode3D(primitive_2, (0.1,))),
    gls.Union(primitive_1, gls.Onion3D(primitive_2, (0.1,))),
    gls.Union(primitive_1, gls.NegOnlyOnion3D(primitive_2, (0.1,))),
    
    gls.Union(primitive_1, gls.Distort3D(primitive_2, (0.1,))),
    gls.Union(primitive_1, gls.Twist3D(primitive_2, (0.5,))),
    gls.Union(primitive_1, gls.Bend3D(gls.Cuboid3D((0.2, 0.4, 0.8,)), (1.1,))),
    # Macros
    gls.ReflectCoords3D(primitive_2, (0.1, 0.2, 0.3)),
    gls.ReflectX3D(primitive_2,),
    gls.ReflectY3D(primitive_2,),
    gls.ReflectZ3D(primitive_2,),
    gls.TranslationSymmetry3D(prim_3, 
        (1, 0, 0), (1.0,), (5.0,)),
    gls.TranslationSymmetryX3D(prim_3, (1.2, ), (5.0,)),
    gls.TranslationSymmetryY3D(prim_3, (1.2, ), (5.0,)),
    gls.TranslationSymmetryZ3D(prim_3, (1.2, ), (7.0,)),
    gls.RotationSymmetryX3D(prim_4, (0.2, ), (5.0,)),
    gls.RotationSymmetryY3D(prim_4, (0.2, ), (5.0,)),
    gls.RotationSymmetryZ3D(prim_4, (0.2, ), (7.0,)),
    # SYSL ops:
    # GeomOnlySmoothUnion

]
# All 3D primitives expressions
all_primitives_expressions = [
    gls.Sphere3D((0.5,)),
    gls.Box3D((0.3, 0.1, 0.2,)),
    gls.Cuboid3D((0.3, 0.1, 0.2,)),
    gls.RoundedBox3D((0.3, 0.1, 0.2,), (0.1,)),
    gls.BoxFrame3D((0.3, 0.5, 0.6,), (0.1,)),
    gls.Torus3D((0.5, 0.1,)),
    gls.CappedTorus3D((0.5,), (0.5,), (0.21,)),
    gls.Link3D(0.5, 0.3, 0.1),
    gls.InfiniteCylinder3D((0.3, 0.1, 0.2,)),
    gls.InfiniteCone3D((0.5,)),
    gls.Cone3D((0.5,), (0.5,)),
    gls.InexactCone3D((0.5,), (0.5,)),
    gls.Plane3D((0.3, 0.1, 0.2,), (0.5,)),
    gls.HexPrism3D((0.3, 0.1,)),
    gls.TriPrism3D((0.3, 0.1,)),
    gls.Capsule3D((0.3, 0.1, 0.2,), (0.3, 0.6, 0.2,), (0.1,)),
    gls.VerticalCapsule3D((0.5,), (0.1,)),
    gls.CappedCylinder3D((0.5,), (0.1,)),
    gls.Cylinder3D((0.5,), (0.1,)),
    gls.ArbitraryCappedCylinder3D((0.3, 0.1, 0.2,), (0.8, 0.1, 0.6,), (0.1,)),
    gls.RoundedCylinder3D((0.1,), (0.5,), (0.4,)),
    gls.CappedCone3D((0.1,), (0.3,), (0.4,)),
    gls.ArbitraryCappedCone3D((0.3, 0.1, 0.2,), (0.3, 0.6, 0.2,), (0.3,), (0.1,)),
    gls.SolidAngle3D((0.5,), (0.3,)),
    gls.CutSphere3D((0.5,), (0.2,)),
    gls.CutHollowSphere((0.3,), (0.5,), (0.1,)),
    gls.DeathStar3D((0.5,), (0.35,), (0.5,)),
    gls.RoundCone3D((0.3,), (0.1,), (0.5,)),
    gls.ArbitraryRoundCone3D((0.1, 0.1, 0.2,), (0.3, 0.7, 0.8,), (0.3,), (0.1,)),
    gls.InexactEllipsoid3D((0.3, 0.1, 0.2,)),
    gls.RevolvedVesica3D((0.0, 0.1, 0.2,), (0.3, 0.1, 0.6,), (0.1,)),
    gls.Rhombus3D((0.3,), (0.1,), (0.5,), (0.1,)),
    gls.Octahedron3D((0.5,)),
    gls.InexactOctahedron3D((0.5,)),
    gls.Pyramid3D((0.5,)),
    # Here on its tricky.
    gls.Triangle3D((0.3, 0.1, 0.2,), (0.3, 0.4, 0.2,), (0.1, 0.0, 0.6,)),
    gls.Quadrilateral3D((0.0, 0.1, 0.2,), (0.3, 0.4, 0.2,), (0.1, 0.0, 0.6,), (0.7, 0.1, 0.2,)),
    gls.NoParamCuboid3D(),
    gls.NoParamSphere3D(),
    gls.NoParamCylinder3D(),
    # Not really usable.
    # gls.PlaneV23D((0.3, 0.1, 0.2,), (0.3, 0.1, 0.2,)),
    # gls.VerticalCappedCylinder3D((0.5,), (0.1,)),
    # gls.InexactSuperQuadrics3D((0.3, 0.1, 0.2,), (0.3,), (0.1,)),
    # gls.InexactAnisotropicGaussian3D((0.0, 0.0, 0.0,), (0.1, 0.1, 0.1,), (0.1,)),
    # gls.NullExpression3D(),
]
# All 2D primitives expressions
all_primitives_2d_expressions = [
    gls.Circle2D((0.5,)),
    gls.RoundedBox2D((0.6, 0.8,), (0.05, 0.30, 0.5, 0.25,)),
    gls.Box2D((0.3, 0.1,)),
    gls.Rectangle2D((0.3, 0.1,)),
    gls.OrientedBox2D((0.0, 0.0,), (0.3, 0.1,), (0.05,)),
    gls.Rhombus2D((0.3, 0.1,)),
    gls.Trapezoid2D((0.3,), (0.1,), (0.2,)),
    gls.Parallelogram2D((0.3,), (0.2,), (0.1,)),
    gls.EquilateralTriangle2D((0.5,)),
    gls.IsoscelesTriangle2D((0.3, 0.2,)),
    gls.Triangle2D((0.0, 0.0,), (0.3, 0.0,), (0.15, 0.3,)),
    gls.UnevenCapsule2D((0.3,), (0.2,), (0.4,)),
    gls.RegularPentagon2D((0.5,)),
    gls.RegularHexagon2D((0.5,)),
    gls.RegularOctagon2D((0.5,)),
    gls.Hexagram2D((0.5,)),
    gls.Pentagram2D((0.5,),),
    gls.RegularStar2D((0.8,), (12,), (2,)),
    gls.Pie2D((0.5,), (0.8,)),
    gls.CutDisk2D((0.5,), (0.2,)),
    gls.Arc2D((0.5,), (0.3,), (0.1,)),
    gls.HorseShoe2D((0.5,), (0.3,), (0.1, 0.05,)),
    gls.Vesica2D((0.5,), (0.3,)),
    gls.OrientedVesica2D((0.0, 0.0,), (0.3, 0.1,), (0.1,)),
    gls.Moon2D((0.3,), (0.5,), (0.4,)),
    gls.RoundedCross2D((0.3,)),
    gls.Egg2D((0.3,), (0.1,), (0.2,), (0.4,)),
    gls.Heart2D(),
    gls.Cross2D((0.3, 0.1,), (0.05,)),
    gls.RoundedX2D((0.3,), (0.05,)),
    gls.Ellipse2D((0.3, 0.1,)),
    gls.BlobbyCross2D((0.3,)),
    gls.Tunnel2D((0.3, 0.1,)),
    gls.Stairs2D((0.3, 0.1,), 5),
    gls.QuadraticCircle2D(),
    gls.CoolS2D(),
    gls.CircleWave2D((0.5,), (0.3,)),
    gls.Segment2D((0.0, 0.0,), (0.3, 0.1,)),
    # These dont make sense for extrusion.
    # gls.Parabola2D((0.5,)),
    # gls.ParabolaSegment2D((0.3,), (0.1,)),
    # gls.Hyperbola2D((0.5,), (0.3,)),
    # gls.QuadraticBezierCurve2D((0.0, 0.0,), (0.3, 0.3,), (0.6, 0.0,)),
    # These don't have default_spec, so skipping for now:
    # Supported in Migumi currently. General Support TBD.
    # gls.PolyArc2D(((0.0, 0.0, 0.0,), (0.3, 0.1, 0.0,), (0.6, 0.0, 0.0,))),
    # gls.Polygon2D(((0.0, 0.0,), (0.3, 0.0,), (0.15, 0.3,))),
    
]
all_primitives_2d_expressions = [gls.SimpleExtrusion3D(x, (0.5,)) for x in all_primitives_2d_expressions]

all_expressions = all_primitives_2d_expressions + all_primitives_expressions + comb_exprs
# Test all Transforms. 
# Test Macros
# Test mix

In [ ]:
SEL_INDEX = np.random.randint(0, len(all_expressions))

# cur_expr = gls.Torus3D((0.5, 0.1,))# all_expressions[SEL_INDEX]
cur_expr = all_expressions[SEL_INDEX]
print(cur_expr)

mode_list = [
    ["v1", sls.MatSolidV1(cur_expr, sls.MaterialV1((2.0,)))],
    ["v2", sls.MatSolidV2(cur_expr, sls.MaterialV2((1.0, 0.0, 0.0)))],
    ["v3", sls.MatSolidV3(cur_expr, sls.MaterialV3(
        (1.0, 0.0, 0.0),
        (0.0, 1.0, 0.0),
        (1.0,), (0.2,), (0.2),
    ))],
    ["v3", sls.MatSolidV3(cur_expr, sls.NonEmissiveMaterialV3(
        (1.0, 0.0, 0.0),
        (1.0,), (0.2,), (0.2),
        ))],
    ["v3", sls.MatSolidV3(cur_expr, sls.MatRefV3("MatWood"))],
    ["v4", sls.MatSolidV4(cur_expr, sls.MaterialV4(
        (1.0, 0.0, 0.0),
        (0.0, 1.0, 0.0), (0.0, 1.0, 0.0)))],
    ["v4", sls.MatSolidV4(cur_expr, sls.MatRefV4("MatWood"))],
    ["v4", sls.MatSolidV4(cur_expr, 
        sls.MatMixV4(
            sls.MatRefV4("MatFloor"),
            sls.MatRefV4("MatWood"),
            (0.5,)))],
    ["v5", sls.MatSolidV4(cur_expr, 
        sls.MaterialV4(
        (1.0, 0.0, 0.0),
        (0.0, 1.0, 0.0), (0.0, 1.0, 0.0)))],
    ["v6", sls.MatSolidV2(cur_expr, sls.MaterialV2((1.0, 0.0, 0.0)))],

]
F_IND = -3
cur_mode, scene_with_material = mode_list[F_IND]

settings = {
    "render_mode": cur_mode,
    "variables": {
        "_ADD_FLOOR_PLANE": False,
        "castShadows": True,
        "_AA": 1,
        "_RAYCAST_MAX_STEPS": 200,
        "EDGE_THICKNESS": 0.00,
    },
    "set_to_ubo": False,
    "export_params": False,
}


# scene_with_material = sls.MatSolidV2(
#         gls.Union(
#     scene_with_material,
#     sls.MatSolidV2(
#         # gls.Plane3D((0.0, 1.0, 0.0,), gls.UniformFloat((-1.0,), (0.0,), (1.0,), "sdf")), 
#         gls.Translate3D(gls.Cuboid3D((10000.5, 0.1, 10000.5,),), (0.0, -1.5, 0.0,)),
#         # sls.MatRefV4("MatPlastic"),
#         sls.MaterialV2((1.0, 1.0, 1.0))
#         )
# ), 
#         sls.MaterialV2((1.0, 0.0, 1.0)))

# Get Shader Code
shader_code, uniforms, textures = evaluate_singlepass(scene_with_material, settings=settings)
# TO visualize in a browser:
with open("shader_code.glsl", "w") as f:
    f.write(shader_code)
html_code = create_shader_html(shader_code, uniforms, textures, show_controls=True)
# To visualize inline in jupyter notebook:
with open("test.html", "w") as f:
    f.write(html_code)
jupy_wrapper_html = make_jupyter_compatible_html(html_code)
# display(HTML(jupy_wrapper_html))

In [ ]:
from sysl.shader.evaluate_multipass import evaluate_multipass
from sysl.shader_runtime.generate_shader_html import create_multibuffer_shader_html

F_IND = -1

cur_mode, scene_with_material = mode_list[F_IND]
# scene_with_material = gls.Union(
#     scene_with_material,
#     sls.MatSolidV2(
#         # gls.Plane3D((0.0, 1.0, 0.0,), gls.UniformFloat((-1.0,), (0.0,), (1.0,), "sdf")), 
#         gls.Translate3D(gls.Cuboid3D((10000.5, 0.1, 10000.5,),), (0.0, -1.5, 0.0,)),
#         # sls.MatRefV4("MatPlastic"),
#         sls.MaterialV2((1.0, 1.0, 1.0))
#         )
# )
settings = {
    "render_mode": cur_mode,
    "variables": {
        "_ADD_FLOOR_PLANE": False,
        "castShadows": True,
        "_AA": 2,
        "_RAYCAST_MAX_STEPS": 200,
        "OUTLINE_THICKNESS": 3.50,
        "DITHER_INTENSITY_FACTOR": 0.5,
        
    },
    "set_to_ubo": False,
    "export_params": False,
}



# Get Shader Code
shader_bundles = evaluate_multipass(scene_with_material, settings=settings, 
    post_process_shader=["all_outline_nobg"]
    )

html_code = create_multibuffer_shader_html(shader_bundles, show_controls=True)
# To visualize inline in jupyter notebook:
with open("test.html", "w") as f:
    f.write(html_code)